# Fonts

https://sagemaker.readthedocs.io/en/stable/frameworks/sklearn/using_sklearn.html#deploy-endpoints-from-model-data

https://sagemaker.readthedocs.io/en/stable/frameworks/sklearn/sagemaker.sklearn.html

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
import boto3
import sagemaker
from sagemaker.session import Session
from sagemaker.sklearn.estimator import SKLearn 
import joblib
from time import strftime, gmtime 

In [ ]:
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
sm_boto3 = boto3.client('sagemaker')


# Upload to S3

In [ ]:
bucket = ''
prefix = ''

In [ ]:
train_path = sess.upload_data(
    path='daily_energy_train.csv',
    bucket=bucket,
    key_prefix=prefix
)
test_path = sess.upload_data(
    path='daily_energy_test.csv',
    bucket=bucket,
    key_prefix=prefix
)

In [ ]:
%%writefile train.py

import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
import joblib
import argparse
import os

def preprocess(df: pd.DataFrame):
    df['Datetime'] = pd.to_datetime(df['Datetime'], format='%Y-%m-%d')
    df.sort_values(by='Datetime', inplace=True)
    df.reset_index(inplace=True, drop=True)
    df = df.resample('D', on='Datetime').sum()
    df.index.freq = 'D'
    return df


def forecast_accuracy(forecast, actual):
    forecast, actual = np.array(forecast), np.array(actual)
    mape = np.mean(np.abs(forecast - actual)/np.abs(actual))*100  # MAPE
    mae = np.mean(np.abs(forecast - actual))    # MAE
    rmse = np.mean((forecast - actual)**2)**.5  # RMSE
    return({'mape(%)':mape, 'mae': mae, 'rmse':rmse})


def model_fn(model_dir):
    clf = joblib.load(os.path.join(model_dir, "model.joblib"))
    return clf

# def input_fn, output_fn, ...


def _list_of_ints(arg):
    # for argparser
    return list(map(int, arg.split(',')))

if __name__=='__main__':
    parser = argparse.ArgumentParser()

    # where to write trained model artifacts
    parser.add_argument(
        "--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR")
    )

    # Train and Test Data
    parser.add_argument(
        "--train", type=str, default=os.path.join(
        os.environ.get("SM_MODEL_DIR"), "daily_energy_train.csv")
    )
    parser.add_argument(
        "--test", type=str, default=os.path.join(
        os.environ.get("SM_MODEL_DIR"), "daily_energy_test.csv")
    )
    
    # Hyperparameters
    parser.add_argument(
        "--order", type=_list_of_ints, default=(3,0,0)
    )
    parser.add_argument(
        "--seasonal_order", type=_list_of_ints, default=(0,1,1,7)
    )

    args, _ = parser.parse_known_args()

    print("\n[INFO] Reading data\n")
    path = os.getcwd()
    train_df = pd.read_csv(args.train)
    test_df = pd.read_csv(args.test)
    
    print("\n[INFO] Processing data")
    train_df = preprocess(train_df)
    test_df = preprocess(test_df)
    
    print('order', type(args.order), args.order)

    print("\n[INFO] Training data with ARIMA")
    model = ARIMA(
        train_df,
        order=args.order,
        seasonal_order=args.seasonal_order
    )
    model = model.fit()

    model_path = os.path.join(args.model_dir, "model.joblib")
    print("\n[INFO] Saving model at", model_path)
    joblib.dump(model, model_path)


    print("\n[INFO] Predicting data")
    predictions = model.forecast(len(test_df))
    # accuracy 
    print('\n[INFO] Model Accuracy:')
    results = forecast_accuracy(predictions, test_df)
    for metric, value in results.items():
        print(f'\t {metric}: {value}')






In [ ]:
# for debug porpuses only, csv data can be passed as an dependency 
sklearn_estimator = SKLearn(
    entry_point='train.py',
    instance_type='ml.m5.4xlarge',
    framework_version='1.0-1',
    base_job_name="timeSeries-train",
    role=role,
    dependencies=['requirements.txt'], 
    hyperparameters={
        'order': "3,0,0",
        'seasonal_order': "0,1,1,7"
    }
)

In [ ]:
sklearn_estimator.fit({'train': train_path, 'test': test_path}, wait=True)

# Model Artifacts: From Trained Job cache to S3

In [ ]:
sklearn_estimator.latest_training_job.wait(logs="None")
artifact = sm_boto3.describe_training_job(
    TrainingJobName=sklearn_estimator.latest_training_job.name
)["ModelArtifacts"]["S3ModelArtifacts"]

print("Model artifact persisted at " + artifact)

# Deploy Sagemaker Endpoint(API) 

In [ ]:
model_name = "Fidelidade-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
model = SKLearnModel(
    name =  model_name,
    model_data=artifact,
    role=get_execution_role(),
    entry_point="script.py",
)

In [ ]:
endpoint_name = "Custom-sklearn-model-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
print("EndpointName={}".format(endpoint_name))

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge",
    endpoint_name=endpoint_name,
)